# GET 324 Laboratory Exercise 10 - Group ME11
## Fresh versus Rotten Orange Image Classification

This notebook prepares a two-class image dataset, trains a MobileNetV3Small transfer-learning model, evaluates it, and saves the files needed by the Streamlit application.

**Class order used throughout the project:** `0 = fresh_orange`, `1 = rotten_orange`.

Run the cells from top to bottom. Read each explanation before running its code cell.

## Step 1: Turn on the Colab GPU

In Colab, select **Runtime > Change runtime type > T4 GPU > Save**. A GPU makes convolutional-network training much faster. The next cell displays the selected device.

In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))

## Step 2: Install and import the required packages

`kagglehub` downloads the public dataset. Scikit-learn performs stratified data splitting and calculates evaluation metrics. TensorFlow and Keras build the model. Pillow checks image files, while Matplotlib and Seaborn create figures.

In [ ]:
!pip -q install kagglehub scikit-learn seaborn

In [ ]:
import hashlib
import json
import random
import shutil
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['fresh_orange', 'rotten_orange']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print('The experiment seed is', SEED)

## Step 3: Download the source dataset

The source is the **Fruits Fresh and Rotten for Classification** dataset on Kaggle. It contains fresh and rotten apple, banana and orange folders. This project keeps only the fresh-orange and rotten-orange images. If Kaggle requests authentication, follow the prompt and use an API token created in your Kaggle account settings.

In [ ]:
DATASET_HANDLE = 'sriramr/fruits-fresh-and-rotten-for-classification'

try:
    downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
except Exception as error:
    print('Automatic download needs Kaggle authentication:', error)
    kagglehub.login()
    downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))

print('Dataset downloaded to:', downloaded_path)

## Step 4: Locate, validate and copy the two required classes

The code searches the downloaded directory for the fresh-orange and rotten-orange folders. It supports the folder names used by the selected Kaggle dataset. Every image is opened with Pillow to reject damaged files. A SHA-256 hash prevents an identical file from entering the dataset twice, which reduces data leakage.

In [ ]:
SOURCE_FOLDER_ALIASES = {
    'fresh_orange': {'freshoranges', 'fresh_orange', 'fresh orange'},
    'rotten_orange': {'rottenoranges', 'rotten_orange', 'rotten orange'},
}


def locate_class_directories(root: Path, class_name: str) -> list[Path]:
    accepted_names = SOURCE_FOLDER_ALIASES[class_name]
    matches = [
        path for path in root.rglob('*')
        if path.is_dir() and path.name.strip().lower() in accepted_names
    ]
    if not matches:
        raise FileNotFoundError(
            f'No source folder for {class_name!r} was found in {root}. '
            f'Expected one of: {sorted(accepted_names)}'
        )
    return matches


CLEAN_DATA_DIR = Path('/content/fresh_rotten_orange_data')
if CLEAN_DATA_DIR.exists():
    shutil.rmtree(CLEAN_DATA_DIR)

valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
seen_hashes = set()
copy_summary = {}

for class_name in CLASS_NAMES:
    source_dirs = locate_class_directories(downloaded_path, class_name)
    target_dir = CLEAN_DATA_DIR / class_name
    target_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    rejected = 0

    for source_dir in source_dirs:
        for source_file in sorted(source_dir.rglob('*')):
            if not source_file.is_file() or source_file.suffix.lower() not in valid_extensions:
                continue
            try:
                with Image.open(source_file) as image:
                    image.verify()
                file_hash = hashlib.sha256(source_file.read_bytes()).hexdigest()
                if file_hash in seen_hashes:
                    rejected += 1
                    continue
                seen_hashes.add(file_hash)
                destination = target_dir / f'{class_name}_{copied:04d}{source_file.suffix.lower()}'
                shutil.copy2(source_file, destination)
                copied += 1
            except (UnidentifiedImageError, OSError, ValueError):
                rejected += 1

    copy_summary[class_name] = {
        'copied': copied,
        'rejected': rejected,
        'sources': [str(path) for path in source_dirs],
    }

print(json.dumps(copy_summary, indent=2))
assert all(copy_summary[name]['copied'] >= 30 for name in CLASS_NAMES), 'Too few valid images were found.'

## Step 5: Inspect class counts and sample images

Visual inspection helps detect wrong labels, drawings, damaged files or unrelated animals before training. The bar chart also shows whether one class has many more images than the other.

In [ ]:
class_files = {
    name: sorted([path for path in (CLEAN_DATA_DIR / name).iterdir() if path.is_file()])
    for name in CLASS_NAMES
}

counts = {name: len(paths) for name, paths in class_files.items()}
print('Usable image counts:', counts)

sns.barplot(x=list(counts.keys()), y=list(counts.values()), hue=list(counts.keys()), legend=False)
plt.title('Number of Valid Images per Class')
plt.ylabel('Images')
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, class_name in enumerate(CLASS_NAMES):
    sample_paths = random.sample(class_files[class_name], min(4, len(class_files[class_name])))
    for column, image_path in enumerate(sample_paths):
        axes[row, column].imshow(Image.open(image_path).convert('RGB'))
        axes[row, column].set_title(class_name.title())
        axes[row, column].axis('off')
plt.tight_layout()
plt.show()

## Step 6: Create stratified training, validation and test sets

The split is 70% training, 15% validation and 15% testing. Stratification keeps nearly the same fresh-to-rotten-orange ratio in every split. The model learns from the training set, training decisions are checked with the validation set, and the untouched test set gives the final performance estimate.

In [ ]:
all_paths = []
all_labels = []
for label, class_name in enumerate(CLASS_NAMES):
    all_paths.extend([str(path) for path in class_files[class_name]])
    all_labels.extend([label] * len(class_files[class_name]))

all_paths = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)

train_paths, temporary_paths, train_labels, temporary_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=all_labels,
)
validation_paths, test_paths, validation_labels, test_labels = train_test_split(
    temporary_paths,
    temporary_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_labels,
)

for split_name, labels in {
    'training': train_labels,
    'validation': validation_labels,
    'testing': test_labels,
}.items():
    values, frequencies = np.unique(labels, return_counts=True)
    distribution = {CLASS_NAMES[value]: int(count) for value, count in zip(values, frequencies)}
    print(f'{split_name.title():10s}: {len(labels):4d} images -> {distribution}')

## Step 7: Build efficient TensorFlow input pipelines

Each file is decoded as a three-channel colour image, resized to 224 by 224 pixels and grouped into batches. Caching avoids repeated disk reads, while prefetching prepares the next batch while the GPU processes the current one.

In [ ]:
def decode_and_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)


def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, training=True)
validation_ds = make_dataset(validation_paths, validation_labels)
test_ds = make_dataset(test_paths, test_labels)

images, labels = next(iter(train_ds))
print('Image batch shape:', images.shape)
print('Label batch shape:', labels.shape)

## Step 8: Define augmentation and the transfer-learning model

Small random flips, rotations, zooms and contrast changes create realistic training variations and reduce overfitting. MobileNetV3Small already learned useful visual features from ImageNet. Its original classifier is removed, the convolutional base is frozen, and a new sigmoid output is added for fresh-versus-rotten-orange classification.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal', seed=SEED),
        tf.keras.layers.RandomRotation(0.08, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name='data_augmentation',
)

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True,
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name='image')
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = tf.keras.layers.Dropout(0.30, name='dropout')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='rotten_orange_probability')(x)
model = tf.keras.Model(inputs, outputs, name='me11_fresh_rotten_orange_classifier')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

model.summary()
print('Trainable parameters:', sum(np.prod(weight.shape) for weight in model.trainable_weights))

## Step 9: Train the new classification head

During this phase, only the newly added output layers learn. Early stopping prevents unnecessary epochs after validation AUC stops improving. Model checkpointing preserves the best model instead of merely keeping the last epoch.

In [ ]:
MODEL_PATH = Path('/content/fresh_rotten_orange_model.keras')

feature_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history_feature = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    callbacks=feature_callbacks,
)

best_feature_auc = max(history_feature.history['val_auc'])
print('Best feature-extraction validation AUC:', round(best_feature_auc, 4))

## Step 10: Fine-tune the upper convolutional layers

Fine-tuning adapts higher-level ImageNet features to fresh and rotten orange markings. Only the last 30 layers are opened, Batch Normalisation layers stay frozen, and a very small learning rate is used to avoid damaging the pretrained features. The saved file is replaced only if validation AUC improves.

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
base_model = next(
    layer for layer in model.layers
    if isinstance(layer, tf.keras.Model) and 'mobilenetv3' in layer.name.lower()
)
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

fine_tune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        initial_value_threshold=best_feature_auc,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
]

history_fine = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=fine_tune_callbacks,
)

## Step 11: Plot the learning curves

Training and validation curves reveal learning behaviour. A widening gap, where training improves while validation worsens, is evidence of overfitting.

In [ ]:
def combine_metric(metric_name):
    return history_feature.history[metric_name] + history_fine.history[metric_name]

epochs = range(1, len(combine_metric('loss')) + 1)
fine_tune_start = len(history_feature.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(epochs, combine_metric('loss'), label='Training loss')
axes[0].plot(epochs, combine_metric('val_loss'), label='Validation loss')
axes[0].axvline(fine_tune_start + 0.5, color='black', linestyle='--', label='Fine-tuning begins')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, combine_metric('accuracy'), label='Training accuracy')
axes[1].plot(epochs, combine_metric('val_accuracy'), label='Validation accuracy')
axes[1].axvline(fine_tune_start + 0.5, color='black', linestyle='--', label='Fine-tuning begins')
axes[1].set_title('Accuracy Curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

## Step 12: Select a decision threshold with validation data

The model outputs the probability of the positive class, rotten_orange. The usual threshold is 0.50. This cell checks several validation thresholds and selects the one with the highest F1 score. The test set is not used for this decision.

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
validation_probabilities = best_model.predict(validation_ds, verbose=0).reshape(-1)
thresholds = np.arange(0.20, 0.81, 0.01)
validation_f1_scores = [
    f1_score(validation_labels, (validation_probabilities >= threshold).astype(int), zero_division=0)
    for threshold in thresholds
]
BEST_THRESHOLD = float(thresholds[int(np.argmax(validation_f1_scores))])

plt.figure(figsize=(8, 4))
plt.plot(thresholds, validation_f1_scores)
plt.axvline(BEST_THRESHOLD, color='red', linestyle='--', label=f'Best = {BEST_THRESHOLD:.2f}')
plt.xlabel('Rotten-orange probability threshold')
plt.ylabel('Validation F1 score')
plt.title('Decision Threshold Selection')
plt.legend()
plt.show()

print('Selected threshold:', round(BEST_THRESHOLD, 2))
print('Validation F1 at selected threshold:', round(max(validation_f1_scores), 4))

## Step 13: Evaluate once on the untouched test set

Accuracy measures total correct predictions. Precision asks how often a predicted rotten_orange is truly a rotten_orange. Recall asks how many real rotten_oranges were detected. F1 balances precision and recall. The confusion matrix shows correct and incorrect predictions for both classes.

In [ ]:
test_probabilities = best_model.predict(test_ds, verbose=0).reshape(-1)
test_predictions = (test_probabilities >= BEST_THRESHOLD).astype(int)

test_accuracy = float(np.mean(test_predictions == test_labels))
test_precision = precision_score(test_labels, test_predictions, zero_division=0)
test_recall = recall_score(test_labels, test_predictions, zero_division=0)
test_f1 = f1_score(test_labels, test_predictions, zero_division=0)

print(f'Test accuracy : {test_accuracy:.4f}')
print(f'Test precision: {test_precision:.4f}')
print(f'Test recall   : {test_recall:.4f}')
print(f'Test F1 score : {test_f1:.4f}')
print('\nClassification report:\n')
print(classification_report(test_labels, test_predictions, target_names=CLASS_NAMES, digits=4, zero_division=0))

matrix = confusion_matrix(test_labels, test_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt='d', cmap='YlOrBr', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted class')
plt.ylabel('Actual class')
plt.title('Test Confusion Matrix')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## Step 14: Inspect sample predictions and errors

A metric alone does not explain model behaviour. These images show whether errors are related to pose, distance, lighting, obstruction or background.

In [ ]:
error_indices = np.where(test_predictions != test_labels)[0]
display_indices = error_indices[:8]
if len(display_indices) == 0:
    display_indices = np.arange(min(8, len(test_paths)))

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
axes = axes.flatten()
for axis in axes:
    axis.axis('off')
for axis, index in zip(axes, display_indices):
    image = Image.open(test_paths[index]).convert('RGB')
    actual = CLASS_NAMES[int(test_labels[index])]
    predicted = CLASS_NAMES[int(test_predictions[index])]
    rotten_orange_probability = test_probabilities[index]
    confidence = rotten_orange_probability if predicted == 'rotten_orange' else 1 - rotten_orange_probability
    axis.imshow(image)
    axis.set_title(f'Actual: {actual}\nPredicted: {predicted} ({confidence:.1%})')
    axis.axis('off')
plt.tight_layout()
plt.show()

## Step 15: Save the deployment files

The `.keras` file contains the complete trained network. `model_info.json` records the class order, input size and selected threshold so that the Streamlit application uses exactly the same settings.

In [ ]:
final_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
final_model.save('/content/fresh_rotten_orange_model.keras')

model_info = {
    'class_names': CLASS_NAMES,
    'image_size': list(IMAGE_SIZE),
    'threshold': BEST_THRESHOLD,
    'minimum_confidence': 0.75,
    'model_name': 'MobileNetV3Small transfer learning',
    'positive_class': 'rotten_orange',
    'test_metrics': {
        'accuracy': test_accuracy,
        'precision': float(test_precision),
        'recall': float(test_recall),
        'f1_score': float(test_f1),
    },
    'dataset_source': 'Kaggle: sriramr/fruits-fresh-and-rotten-for-classification',
}

with open('/content/model_info.json', 'w', encoding='utf-8') as file:
    json.dump(model_info, file, indent=2)

print('Saved model:', Path('/content/fresh_rotten_orange_model.keras').stat().st_size / (1024**2), 'MB')
print(json.dumps(model_info, indent=2))

## Step 16: Back up the results to Google Drive

Colab storage is temporary. Mount Google Drive and copy the model, configuration and evaluation figures before the session ends.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/GET324_ME11_Fresh_Rotten_Orange')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

for file_name in [
    'fresh_rotten_orange_model.keras',
    'model_info.json',
    'training_curves.png',
    'confusion_matrix.png',
]:
    source = Path('/content') / file_name
    if source.exists():
        shutil.copy2(source, DRIVE_OUTPUT / file_name)

print('Files copied to:', DRIVE_OUTPUT)
print([path.name for path in DRIVE_OUTPUT.iterdir()])

## Step 17: Download the two deployment files

Download both files and place them in the same GitHub repository folder as `app.py`. The Streamlit app cannot make predictions without the trained `.keras` file.

In [ ]:
from google.colab import files

files.download('/content/fresh_rotten_orange_model.keras')
files.download('/content/model_info.json')

## Final interpretation checklist

Before accepting the result, record:

1. Number of valid fresh and rotten orange images.
2. Training, validation and test counts.
3. Best validation AUC and selected decision threshold.
4. Test accuracy, precision, recall and F1 score.
5. The confusion matrix and at least two difficult examples.
6. Any signs of overfitting in the learning curves.
7. Limitations, including small dataset size and out-of-distribution images.

After this notebook is complete, test `app.py` locally, push the repository to GitHub, and deploy it through Streamlit Community Cloud.